# Modular Exponentiation

The `modular_exponentiate` function raises a classical integer `a` to the power of a quantum number `p` modulo classical integer `modulus`, times a quantum number `x`.

The function performs:
$$|x\rangle |p\rangle = |(a^{p} \mod \text{modulus})\cdot x\rangle | p\rangle$$

Specifically if at the input $x=1$, at the output $x=a^{p} \mod \text{modulus}$.

## Example

This example generates a quantum program that initializes a `power` variable with a uniform superposition, and exponentiate the classical value `A` with `power` as the exponent, in superposition. The result is calculated inplace to the variable `x` module `N`.

Notice that `x` should have size of at least $\lceil(log_2(N) \rceil$, so it is first allocated with a fixed size, then initialized with the value '1'.

In [1]:
import numpy as np

from classiq import *
from classiq.qmod.symbolic import ceiling, log

# constants
N = 5
A = 4


@qfunc
def main(power: Output[QNum], x: Output[QNum]) -> None:
    allocate(ceiling(log(N, 2)), x)
    x ^= 1

    # initialize a uniform superposition of powers
    allocate(3, power)
    hadamard_transform(power)

    modular_exponentiate(modulus=N, a=A, x=x, p=power)


qprog = synthesize(main)

In [2]:
result = sample(qprog)
result

Submitting job to simulator


Job: https://platform.classiq.io/jobs/172f78cd-2b9c-408e-94a5-f7cf1a25f11d


,power,x,counts,probability,bitstring
0,7,4,284,0.138672,100111
1,0,1,265,0.129395,001000
2,2,1,263,0.128418,001010
3,3,4,263,0.128418,100011
4,5,4,257,0.125488,100101
5,4,1,249,0.121582,001100
6,1,4,236,0.115234,100001
7,6,1,231,0.112793,001110


Verify all results are as expected:

In [3]:
assert np.all(result.apply(lambda row: row["x"] == (A ** row["power"] % N), axis=1))